# Prueba YOLO11s · 20 épocas

Entrenamiento en Colab desde VS Code. Mantiene train/valid, resolución 640, batch 8, seed 42 y 20 épocas. No evalúa test ni cambia la política vigente.

Seleccionar **Colab → nuevo servidor GPU** y ejecutar las celdas por orden.


In [2]:
import sys, torch
print("Python:",sys.version.split()[0])
print("PyTorch:",torch.__version__)
assert torch.cuda.is_available(), "Selecciona un servidor Colab con GPU."
print("GPU:",torch.cuda.get_device_name(0))


Python: 3.13.16
PyTorch: 2.11.0+cu130
GPU: Tesla T4


## Preparar
Subir la carpeta **data/colab/yolo11s_upload_parts** mediante **Upload to Colab** desde el Explorador de VS Code. Solo contiene datos públicos de train/valid, código y supuestos del experimento; no incluye claves ni test.


In [3]:
from pathlib import Path
import hashlib, json, subprocess, zipfile
parts_dir=Path("/content/yolo11s_upload_parts")
info=json.loads((parts_dir/"transfer_info.json").read_text())
parts=sorted(parts_dir.glob("yolo11s_upload.part*"))
assert len(parts)==info["parts"], "La subida no ha terminado."
archive=Path("/content/yolo11s_trial_upload.zip")
with archive.open("wb") as target:
    for part in parts: target.write(part.read_bytes())
assert hashlib.sha256(archive.read_bytes()).hexdigest()==info["archive_sha256"]
cloud_root=Path("/content/car_damage_yolo11s_trial")
assert not (cloud_root/"results/latest_training.json").exists(), "Este experimento ya se ha iniciado; no lo sobrescribas."
with zipfile.ZipFile(archive) as z:
    for name in z.namelist():
        assert Path("/content") in (Path("/content")/name).resolve().parents
    z.extractall("/content")
manifest=json.loads((cloud_root/"transfer_manifest.json").read_text())
for name,expected in manifest.items():
    assert hashlib.sha256((cloud_root/name).read_bytes()).hexdigest()==expected, name
assert not (cloud_root/"data/car-damage-v1/test").exists()
print("Archivos verificados. Preparando Ultralytics 8.4.173...",flush=True)
subprocess.run([sys.executable,"-m","pip","install","--quiet","ultralytics==8.4.173"],check=True)
print("Preparado; solo train y validación.")


Archivos verificados. Preparando Ultralytics 8.4.173...
Preparado; solo train y validación.


## Entrenar y evaluar validación
Ejecuta una prueba nueva con YOLO11s preentrenado. Conserva las veinte épocas y elige el checkpoint por validación. La comparación económica usa el presupuesto actual de 92 minutos.


In [4]:
import os, signal, subprocess, sys
from pathlib import Path
cloud_root=Path("/content/car_damage_yolo11s_trial")
log_path=cloud_root/"experiment.log"
with log_path.open("w") as log:
    process=subprocess.Popen([sys.executable,"-u","tools/run_validation_experiment.py"],cwd=cloud_root,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,start_new_session=True)
    try:
        for line in process.stdout:
            log.write(line);log.flush()
            if any(word in line for word in ("PILOT epoch=","PILOT COMPLETE","EXPERIMENT","photos","Error","Traceback","WARNING")):
                print(line.strip(),flush=True)
        code=process.wait()
    except KeyboardInterrupt:
        os.killpg(process.pid,signal.SIGINT);process.wait();raise
if code:
    print("".join(log_path.read_text().splitlines(keepends=True)[-45:]))
    raise RuntimeError(f"Experimento interrumpido: código {code}")
print("EXPERIMENT COMPLETE. Preparar la descarga con la siguiente celda.")


EXPERIMENT ENVIRONMENT {"python": "3.13.16", "torch": "2.11.0+cu130", "ultralytics": "8.4.173", "cuda": "13.0", "gpu": "Tesla T4", "pretrained_sha256": "85a76fe86dd8afe384648546b56a7a78580c7cb7b404fc595f97969322d502d5", "test_evaluated": false}
PILOT epoch=1/20 val_mAP50=0.5451 val_mAP50_95=0.2486
PILOT epoch=2/20 val_mAP50=0.2785 val_mAP50_95=0.1229
PILOT epoch=3/20 val_mAP50=0.4140 val_mAP50_95=0.1972
PILOT epoch=4/20 val_mAP50=0.5589 val_mAP50_95=0.3128
PILOT epoch=5/20 val_mAP50=0.6074 val_mAP50_95=0.3198
PILOT epoch=6/20 val_mAP50=0.6698 val_mAP50_95=0.3659
PILOT epoch=7/20 val_mAP50=0.7283 val_mAP50_95=0.4194
PILOT epoch=8/20 val_mAP50=0.7461 val_mAP50_95=0.4307
PILOT epoch=9/20 val_mAP50=0.7633 val_mAP50_95=0.4242
PILOT epoch=10/20 val_mAP50=0.7705 val_mAP50_95=0.4676
PILOT epoch=11/20 val_mAP50=0.7889 val_mAP50_95=0.4640
PILOT epoch=12/20 val_mAP50=0.8111 val_mAP50_95=0.4817
PILOT epoch=13/20 val_mAP50=0.8207 val_mAP50_95=0.4956
PILOT epoch=14/20 val_mAP50=0.8428 val_mAP50_95=0

## Descargar resultados
Ejecutar y descargar **/content/yolo11s_trial_results.zip** desde **Colab → Contents**. Incluye el modelo, las 20 épocas, métricas y predicciones de validación con sus hashes.


In [5]:
from pathlib import Path
import json,hashlib,zipfile
cloud_root=Path("/content/car_damage_yolo11s_trial")
state=json.loads((cloud_root/"results/latest_training.json").read_text())
assert state["status"]=="completed" and state["completed_epochs"]==20
result_dir=cloud_root/state["directory"]
assert json.loads((result_dir/"summary.json").read_text())["test_evaluated"] is False
assert (result_dir/"architecture_comparison.json").is_file(), "Falta terminar la comparación de validación."
files=[p for p in result_dir.rglob("*") if p.is_file()]
files+=[cloud_root/state["weights"],cloud_root/"experiment.log"]
manifest={p.relative_to(cloud_root).as_posix():hashlib.sha256(p.read_bytes()).hexdigest() for p in files}
result_zip=Path("/content/yolo11s_trial_results.zip")
with zipfile.ZipFile(result_zip,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=1) as z:
    for p in files:z.write(p,p.relative_to(cloud_root).as_posix())
    z.writestr("experiment_export_manifest.json",json.dumps({"run_id":result_dir.name,"files":manifest},indent=2))
print("Descargar:",result_zip,"·",round(result_zip.stat().st_size/1024**2,1),"MB")


Descargar: /content/yolo11s_trial_results.zip · 21.8 MB
